# MLP 구현 - 실습

---

## 학습 개요

1. **학습 주제**
  - **PyTorch 기본 구성 요소**: 텐서(Tensor), 자동 미분(Autograd), 모듈(nn.Module), 손실 함수, 옵티마이저의 역할과 사용법
  - **데이터 처리 파이프라인**: Dataset과 DataLoader를 활용한 배치 처리 및 데이터 전처리 흐름
  - **학습 루프 구성**: 순전파(forward), 역전파(backward), 파라미터 업데이트 단계로 구성된 학습·평가 루프 설계

2. **학습 목표**
  - PyTorch의 **텐서 연산과 Autograd** 메커니즘을 이해하고 간단한 **경사하강법을 구현**할 수 있다.
  - **nn.Module**을 상속받아 MLP 모델을 **정의**하고 `forward` 메서드를 **구현**할 수 있다.
  - **TensorDataset**과 **DataLoader**를 활용해 배치 단위로 데이터를 **로드**하고 **전처리**할 수 있다.
  - **손실 함수**(CrossEntropyLoss)와 **옵티마이저**(Adam)를 사용해 학습 루프를 **작성**하고 모델을 **학습**시킬 수 있다.
  - 학습 중 **train()/eval()** 모드를 **전환**하고 **Early Stopping**을 적용하여 최적 모델을 **저장·불러오기**할 수 있다.

3. **핵심 개념**
  - **torch.Tensor**: PyTorch의 기본 자료구조로, NumPy 배열과 유사하지만 GPU 가속과 자동 미분을 지원하는 다차원 배열
  - **Autograd**: 텐서 연산 그래프를 자동으로 추적하고 `.backward()` 호출 시 역전파를 수행하여 기울기를 계산하는 자동 미분 엔진
  - **nn.Module**: PyTorch 모델의 기본 클래스로, 파라미터 관리와 forward 메서드를 통해 신경망을 정의하는 인터페이스
  - **nn.Sequential**: 여러 레이어를 순서대로 실행하도록 묶어주는 컨테이너로, forward 메서드를 자동으로 구성
  - **TensorDataset**: PyTorch 텐서들을 하나의 데이터셋으로 묶어주는 클래스로, 인덱싱을 통해 샘플 단위로 접근 가능
  - **DataLoader**: Dataset을 배치 단위로 로드하고 shuffle·병렬 처리를 지원하는 이터레이터 클래스
  - **CrossEntropyLoss**: 다중 클래스 분류에서 예측 확률 분포와 실제 레이블 간의 차이를 측정하는 손실 함수
  - **Optimizer (Adam)**: 손실 함수의 기울기를 바탕으로 모델 파라미터를 업데이트하는 최적화 알고리즘

4. **선행 지식**
  - 파이썬 기초 문법(클래스, 함수, 반복문)
  - NumPy 배열 연산 및 브로드캐스팅
  - 머신러닝 기초(지도학습, 손실 함수, 경사하강법)
  - 신경망 기초(퍼셉트론, 활성화 함수, 역전파)

## 실습 구성

1. **학습 방향**

  - **실습 구성 방식**: 각 단계별로 TODO 영역을 채우며 학습자가 직접 구현

  - **Required Package**:
    ```
    python>=3.11
    numpy>=2.0.0
    matplotlib>=3.8.0
    seaborn>=0.13.2
    scikit-learn>=1.4.2
    torch>=2.0.0
    ```

  - **Step 요약 (진행 흐름)**
    - **Step 1: PyTorch 텐서와 Autograd 이해 (예상 10분)** - 텐서 생성과 연산, requires_grad를 통한 자동 미분, 간단한 선형회귀로 경사하강법 체험
    - **Step 2: PyTorch 핵심 구성 요소 학습 (예상 10분)** - nn.Module, 손실 함수, 옵티마이저를 활용한 학습 파이프라인 구성 및 자동화
    - **Step 3: Dataset과 DataLoader 활용 (예상 15분)** - TensorDataset으로 데이터 묶기, DataLoader로 배치 처리 및 shuffle 적용
    - **Step 4: MLP 모델 구현 및 학습 (예상 20분)** - nn.Sequential로 다층 퍼셉트론 정의, train/eval 모드 전환, Early Stopping 적용
    - **총 예상 소요시간**: 약 55분 (1시간 이내)

2. **데이터셋 개요 및 저작권 정보**
  - **데이터셋 명**: sklearn `load_digits`
  - **데이터셋 개요**: 8×8 화소로 축소된 손글씨 숫자 이미지 1,797개와 레이블(0–9)로 구성된 다중 분류용 데이터셋
  - **사용 목적**: PyTorch 기본 파이프라인을 익히고 MLP 모델을 학습·평가하여 딥러닝 학습 흐름을 실습
  - **저작권/출처**: UCI Machine Learning Repository 공개 도메인 데이터로, scikit-learn에서 재배포 및 학습 목적으로 사용 가능

3. **문제 설명**

  - **문제 개요**: 이 실습은 **PyTorch의 핵심 구성 요소**(텐서, 모듈, 손실 함수, 옵티마이저, 학습 루프)를 이해하고 기초적인 딥러닝 모델을 구현하기 위해 설계되었습니다. 학습자는 **텐서 연산과 자동 미분**을 확인하고, 최종적으로 **MLP 모델을 정의·학습·평가**할 수 있어야 합니다.

  - **요구사항 요약**:
    - **Step 1: PyTorch Tensor**
      - **1줄 요약**: PyTorch 텐서 연산 및 Autograd 동작을 이해한다.
    - **Step 2: 선형회귀**
      - **1줄 요약**: nn.Linear, 손실 함수, 옵티마이저를 활용한 선형회귀를 구현한다.
    - **Step 3: PyTorch Dataset**
      - **TODO 1**: 데이터 분할 및 표준화 - 데이터를 train/valid/test로 분할하고 표준화 수행 (X_valid, X_test 포함)
      - **TODO 2**: PyTorch Dataset 만들기 - TensorDataset을 활용해 배치 처리를 위한 Dataset 구성
      - **1줄 요약**: Dataset과 DataLoader로 배치 처리 파이프라인 구성
    - **Step 4: 학습 루프 진행하기**
      - **TODO 3**: Multi-Layer Perceptron 구현하기 - 기본적인 MLP 모델 정의
      - **TODO 4**: 학습 함수 작성 - DataLoader, Model, Optimizer를 받아 모델을 업데이트하는 학습 코드 작성
      - **TODO 5**: 평가 함수 작성 - 모델과 검증/테스트 데이터로 정확도를 계산하는 평가 코드 작성
      - **1줄 요약**: MLP 모델 정의, 학습/평가 루프 작성, Early Stopping 적용

## Install & Import

In [1]:
# 공통 실습 환경 설치 (최초 1회 실행)
!pip install -q \
    "numpy>=2.0.0" \
    "matplotlib>=3.8.0" \
    "seaborn>=0.13.2" \
    "scikit-learn>=1.4.2"
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu130



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import math
import time
from typing import Tuple

import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

## Step 1: PyTorch 텐서와 Autograd 이해

### Concept Check

오늘은 드디어 가장 기초적인 딥러닝 실습을 진행할 예정입니다. PyTorch 라이브러리를 통해 기본적인 Multi-layer Perceptron을 구현하여 학습하는 방법에 대해서 배워볼 예정입니다. 이번 실습은 모델을 성공적으로 학습시키는 것보다, **PyTorch의 핵심 구성 요소**(텐서, 모듈, 손실 함수, 옵티마이저, 학습 루프)를 이해하고 친해지는 것에 초점을 둡니다.

PyTorch는 NumPy와 사용이 유사합니다. 실제로 사용자들의 혼란을 줄이기 위해 NumPy에 존재하는 같은 기능의 함수를 같은 이름으로 구현해둔 것이 대부분입니다. 하지만 일부 차이가 존재할 수 있기 때문에 정확한 동작을 이해하기 위해서는 API Reference를 항상 참고해주세요.

NumPy의 시작이 `np.array`였다면 PyTorch의 시작은 `torch.Tensor`입니다. 모든 것은 텐서를 기반으로 동작합니다.

In [ ]:
# PyTorch를 사용하기 위해서는 torch를 import 해야합니다.
import torch

x = torch.rand(3, 3)   # 0~1 사이 랜덤 값
y = torch.ones(3, 3)   # 모두 1인 텐서
print("x:\n", x)
print("y:\n", y)
print("x + y:\n", x + y)
print("x @ y.T:\n", x @ y.T)  # 행렬곱

### Concept Check: PyTorch Autograd

가속화가 필수라면 굳이 PyTorch를 쓰지 않고 NumPy를 CUDA에서 사용할 수 있게 만든 CuPy를 쓰는게 낫지 않았을까요? PyTorch를 사용해야하는 가장 큰 이유는 바로 **Autograd** 기능 때문입니다.

Autograd는 PyTorch에서 자동으로 미분을 계산해주는 엔진입니다. 우리가 텐서 연산을 수행하면, PyTorch는 **연산 그래프(computational graph)** 를 만들고, `.backward()` 호출 시 그 그래프를 따라 **역전파(Backpropagation)** 를 실행하여 기울기를 구합니다.

**기본 동작 원리**:
1. `requires_grad=True`로 설정된 텐서에 대해 모든 연산 과정을 추적합니다.
2. 연산이 일어날 때마다 `grad_fn`이라는 연산 노드가 연결되어 그래프가 확장됩니다.
3. `.backward()`를 호출하면, 스칼라 값에서 시작하여 그래프를 따라 거꾸로 기울기를 전파합니다.
4. 계산된 기울기는 `.grad` 속성에 저장됩니다.

**예시**:
$$ y = x^2 + 3x + 1, \quad x, y \in \mathbb{R}^2 $$
$$ z = y_1 + y_2 $$

여기서 스칼라 출력값 $z$는 $x$의 변화에 어떻게 반응할까요? 우리는 이를 $\frac{\partial z}{\partial x}$라고 표기되는 미분으로 표현하고, 역전파를 위해 이 값이 늘 필요합니다. 수식을 조금 더 구체적으로 살펴볼까요?

$$
\frac{\partial z}{\partial x}=\left[ \frac{\partial z}{\partial x_1} , \frac{\partial z}{\partial x_2} \right]
= \left[ \frac{\partial z}{\partial y_1} \frac{\partial y_1}{\partial x_1} ,  \frac{\partial z}{\partial y_2} \frac{\partial y_2}{\partial x_2} \right]
$$

$\frac{\partial z}{\partial y}$는 모두 1이기 때문에 실질적으로 $\frac{\partial y}{\partial x}$만 계산해주면 됩니다. $x$에 대한 $y$의 미분은 손쉽게 $2x+3$이 되는 것을 알기 때문에 최종적으로 미분은 다음과 같이 계산됩니다.

$$
\frac{\partial z}{\partial x}=[2x_1+3, 2x_2+3]
$$

실제로 이렇게 계산되는지 확인해볼까요? 아래 코드를 살펴봅시다. 현재 $x=[2, 3]$인 벡터를 가정하고 연산을 해봅시다. 수식에 따르면 $\frac{\partial z}{\partial x}=[7, 9]$가 계산되어야 합니다.

**핵심 PyTorch 메서드 설명**:
- **torch.tensor(data, requires_grad=True)**: 입력으로 Python 리스트나 NumPy 배열을 받아 PyTorch 텐서로 변환하고 `requires_grad=True`로 설정하면 해당 텐서에 대한 연산 그래프를 추적하여 역전파 시 기울기를 계산
- **Tensor.backward()**: 입력 없이 호출되며 스칼라 텐서에서 계산 그래프를 역방향으로 순회하며 `requires_grad=True`인 모든 텐서의 `.grad` 속성에 기울기를 누적
- **Tensor.grad**: 텐서의 기울기를 저장하는 속성으로, `.backward()` 호출 후 해당 텐서에 대한 손실 함수의 미분값을 출력

In [ ]:
x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x ** 2 + 3 * x + 1
z = y.sum()        # 스칼라

z.backward()       # 역전파 시작
print(x.grad)      # dz/dx 값 출력: [7., 9.]

실제로 잘 계산되었나요? 이를 어떻게 응용해볼 수 있을까요? 선형회귀 모델을 다시 떠올려봅시다. 아주 간단한 모델을 생각해보겠습니다.

$$ y = \theta x + b $$

여기서 $\theta$와 $b$는 학습해야하는 매개변수이고, 실제로 예측해야하는 회귀 타겟변수 $y$, 그리고 독립변수 $x$가 존재한다고 해봅시다. 어떻게 경사하강법을 진행했었는지 다시 떠올려봅시다.

데이터가 `x=2`, `y=4`인 상황이라고 가정합시다.
1. 초기 매개변수값 설정 (e.g. $\theta=3$, $b=1$)
2. 현재 매개변수를 통한 예측값 출력 (e.g. $\hat{y}=3\times2+1=7$)
3. MSE 계산 (e.g. $L=(\hat{y}-y)^2=(7-4)^2=9$)
4. MSE를 기반으로 미분 후 업데이트
  - $\frac{\partial L}{\partial \theta}=2(\hat{y}-y)x=2\times3\times2=12$
  - $\frac{\partial L}{\partial b}=2(\hat{y}-y)=2\times3=6$

복습해보니 기억이 나시나요? 아래 수식으로 실제로 계산이 되는지 확인해봅시다.

In [ ]:
# 파라미터 정의 (requires_grad=True)
weight = torch.tensor([[3.0]], requires_grad=True)
bias   = torch.tensor([[1.0]], requires_grad=True)

# 입력과 목표값
x = torch.tensor([[2.0]])
y_true = torch.tensor([[4.0]])

# 순전파 → 손실 계산
y_pred = x @ weight + bias              # 선형 모델
loss = torch.mean((y_pred - y_true) ** 2)

# 역전파
loss.backward()

weight.grad, bias.grad

실제로 미분값이 잘 나오네요. 여기서 우리가 업데이트를 진행해주는 방식은 현재 매개변수에서 학습률만큼 곱한 미분을 빼주는 것이었습니다. 코드를 마저 마무리해보면

**핵심 PyTorch 메서드 설명**:
- **torch.no_grad()**: 컨텍스트 매니저로 입력 없이 사용되며 내부 블록에서 수행되는 연산의 기울기 추적을 비활성화하여 메모리를 절약하고 파라미터 업데이트 시 그래프가 생성되지 않도록 함
- **Tensor.grad.zero_()**: 입력 없이 호출되며 텐서의 `.grad` 속성을 0으로 초기화하여 기울기 누적을 방지하고 다음 역전파를 위해 준비

In [ ]:
# 파라미터 업데이트
with torch.no_grad():
    # requires_grad=True인 텐서는 inplace operation (+=, -=와 같은)이 불가합니다.
    # 이런 경우 torch.no_grad context를 켜서 작업해주면
    weight -= 0.1 * weight.grad
    bias   -= 0.1 * bias.grad

    # 미분은 덮어쓰는게 아니라 누적 되기 때문에 없애줘야함.
    # weight / bias의 미분을 중간중간 없애줘야함
    weight.grad.zero_()
    bias.grad.zero_()

print(weight, bias)

## Step 2: PyTorch 핵심 구성 요소 학습

### Concept Check

잘 업데이트된걸 확인할 수 있습니다. 하지만 아직은 PyTorch의 장점을 모두 사용하지는 않았습니다. 아무래도 학습에서 제일 중요한 부분은 미분을 계산하는 부분이지만, 그 밖에 많은 부분들은 여전히 수동적으로 업데이트합니다. 위의 코드에서 몇 가지를 자동화해볼 수 있습니다.

1. **Parameter 선언**: 선형모델에서 Feature마다 곱해지는 선형계수를 간단하게 처리할 수 있습니다. 편향항까지 따로 정의하지않고 계산할 수 있습니다.
2. **손실함수 정의**: MSE와 같이 단순한 수식은 함수로 정의하면 좋지만, 새로운 프로젝트가 생기면 거기에 또다시 MSE를 작성해야 합니다. 이는 코드 관리를 하는데 있어 불리한 면이 작용합니다. PyTorch에는 우리가 자주 쓰는 MSELoss 같은 손실함수들이 구현되어 있습니다. import 후 사용하면 됩니다.
3. **Parameter 업데이트**: `torch.optim` 모듈은 우리가 학습해야하는 파라미터를 등록하면, 미분값에 따라 자동으로 업데이트해주는 기능을 가지고 있습니다.

설명만 듣고는 이해가 어렵습니다. 다음 코드를 확인해봅시다.

**핵심 PyTorch 메서드 설명**:
- **nn.Linear(in_features, out_features)**: 입력으로 입력 차원과 출력 차원을 받아 선형 변환 $y = xW^T + b$를 수행하는 레이어를 출력하며, 가중치와 편향을 자동으로 초기화하여 학습 가능한 파라미터로 관리
- **nn.MSELoss()**: 입력으로 예측값과 목표값을 받아 평균 제곱 오차 $\frac{1}{n}\sum(y_{pred} - y_{true})^2$를 계산한 스칼라 텐서를 출력하며, 회귀 문제의 손실 함수로 사용
- **optim.SGD(params, lr)**: 입력으로 최적화할 파라미터 리스트와 학습률을 받아 경사하강법 계열 옵티마이저를 생성하며, `.step()` 호출 시 파라미터를 업데이트합니다. 실제 업데이트 단위는 데이터 배치 구성(Full-batch/Mini-batch/SGD)에 따라 달라질 수 있습니다.
- **optimizer.zero_grad()**: 입력 없이 호출되며 옵티마이저에 등록된 모든 파라미터의 기울기를 0으로 초기화하여 이전 배치의 기울기가 누적되지 않도록 함
- **optimizer.step()**: 입력 없이 호출되며 `.backward()`로 계산된 기울기를 바탕으로 옵티마이저에 등록된 파라미터를 업데이트

In [ ]:
from torch import nn
from torch import optim

# 더미 데이터 준비: y = 2x + 1
X = torch.linspace(0, 10, 100).unsqueeze(1)  # shape: (100, 1)
y_true = 2 * X + 1 + 0.5 * torch.randn_like(X)  # 약간의 노이즈 추가

# 1. Parameter 선언
# 이렇게 하면 weight, bias가 자동으로 준비됩니다.
# 우리는 1차원 변수 x로 1차원 변수 y를 예측하기 때문에 아래와 같이 선언하면 됩니다.
model = nn.Linear(in_features=1, out_features=1)  # 입력 1차원, 출력 1차원

# 2. 손실 함수 정의
# torch.nn에서 불러올 수 있습니다.
criterion = nn.MSELoss()

# 3. Parameter 업데이트
# 현재 사용하고 있는 업데이트 방식은 Mini-batch Gradient Descent입니다.
# 정확한 정의는 batch_size=1일때 Stochastic Gradient Descent, 줄여서 SGD라고 부르는데
# PyTorch에서는 가장 기본적인 미분 기반의 업데이트 방식을 SGD를 선언하여 진행할 수 있습니다.
# 같은 optim.SGD라도 배치 구성에 따라 Full-batch/Mini-batch/SGD 모두로 동작할 수 있습니다.
# 선언할 때는 업데이트 대상인 매개변수와 학습률을 입력해야합니다.
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 4. 학습 루프
epochs = 200
for epoch in range(epochs):
    # 기울기 초기화: 위에서 진행한 weight.grad.zero_()를 자동으로 수행해줍니다.
    # 수행대상은 선언할 때 등록한 모델의 매개변수입니다.
    optimizer.zero_grad()

    # 예측값 만들어주기: model(X)로 가능합니다
    # 이에 대한 자세한 작동방식은 후에 살펴보겠습니다.
    y_pred = model(X)

    # Loss 계산도 마찬가지로 위에서 선언한 손실함수를 활용합니다.
    loss = criterion(y_pred, y_true)

    # Loss를 계산하고 미분을 계산해줍니다.
    loss.backward()

    # 매개변수를 업데이트해줍니다.
    # 위에서 진행한 weight -= lr * weight.grad 부분입니다.
    # 모든 매개변수가 optimizer 안에 등록되어 있기 때문에
    # 매개변수 별로 업데이트 코드를 짜주지 않아도 됩니다.
    optimizer.step()

    if (epoch+1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# 학습 결과 출력
print("\n학습된 가중치와 편향:")
for name, param in model.named_parameters():
    print(f"{name}: {param.data}")

가장 기본적인 PyTorch 학습코드를 완성시켰습니다 🎉. 그 외에 대다수 연산은 `numpy`와 유사한 점이 많기 때문에 크게 어려운 점이 없을겁니다. 추가로 알아야할 사항이 있습니다.

1. **가속화 기기 활용하기**: `torch.Tensor`를 CPU가 아닌 GPU 연산을 사용하려면 `.to` 메소드를 활용해야 합니다. 다음 셀에서 예제로 살펴보겠습니다.
2. **`numpy.array` ↔️ `torch.Tensor`**: 두 라이브러리 사이 데이터를 변환하는 것은 자주 등장하기 때문에 알아두어야합니다.
3. **복잡한 모델 구성하기**: 위에서는 `nn.Linear` 하나로 모델이 완성되었지만, 실제로 수업시간에 배운 Multi-Layer Perceptron 같은 비선형 모델을 구현하기 위해서는 추가적인 모듈이 필요합니다. 이를 하나의 모듈로 결합하는 방법에 대해 배워봅시다.

**핵심 PyTorch 메서드 설명**:
- **torch.cuda.is_available()**: 입력 없이 호출되며 현재 시스템에서 CUDA(GPU)를 사용할 수 있는지 여부를 Boolean으로 출력
- **torch.from_numpy(ndarray)**: 입력으로 NumPy 배열을 받아 동일한 메모리를 공유하는 PyTorch 텐서로 변환하여 출력
- **Tensor.cpu()**: 입력 없이 호출되며 GPU에 있는 텐서를 CPU로 이동시킨 텐서를 출력
- **Tensor.numpy()**: 입력 없이 호출되며 CPU에 있는 PyTorch 텐서를 NumPy 배열로 변환하여 출력 (GPU 텐서는 먼저 `.cpu()` 호출 필요)

In [ ]:
# 1. 가속화기기 활용하기
cuda_on = torch.cuda.is_available()
print(f"CUDA가 사용이 가능한가요?: {cuda_on}")

cpu_tensor = torch.eye(4)
DEVICE = torch.device("cuda")
gpu_tensor = cpu_tensor.to(DEVICE)

print(f"CPU Tensor: {cpu_tensor}")
print(f"GPU Tensor: {gpu_tensor}")

In [ ]:
# 2. NumPy and Torch
import numpy as np

# numpy to torch
np_array = np.eye(4)
t_from_numpy = torch.from_numpy(np_array)
print(np_array, t_from_numpy)

# torch to numpy
# .numpy를 호출하면 쉽게 변환 가능합니다.
# 하지만 텐서가 CUDA에 올라가 있다면 CPU로 변환하고 작동시켜줘야합니다.
t_tensor = torch.eye(4).to(DEVICE)
np_from_tensor = t_tensor.cpu().numpy()
print(t_tensor, np_from_tensor)

### Concept Check: Multi-Layer Perceptron 구성하기

`nn.Linear`로도 해결이 가능하면 좋겠지만, 일반적으로 인공신경망은 이러한 선형레이어와 몇 가지 비선형 함수를 추가하여 구성됩니다. 복잡한 구성들을 하나의 모델로 합쳐서 사용하는 방법을 배워봅시다.

**핵심 PyTorch 메서드 설명**:
- **nn.Module**: PyTorch 모델의 기본 클래스로, 입력으로 `__init__`에서 레이어를 정의하고 `forward` 메서드에서 순전파 로직을 구현하면 자동으로 파라미터 관리와 훅 등록 기능을 제공
- **nn.Sequential(*layers)**: 입력으로 레이어들을 순서대로 받아 이를 하나의 모듈로 묶어 순차 실행하는 컨테이너를 출력하며, `forward` 메서드를 자동 생성
- **nn.ReLU()**: 입력 텐서를 받아 각 원소에 대해 $\max(0, x)$를 적용한 텐서를 출력하며, 음수 값을 0으로 만들어 비선형성을 추가하는 활성화 함수로 사용
- **Module.forward(x)**: 입력 텐서 x를 받아 모델의 순전파 로직을 수행한 출력 텐서를 반환하며, `model(x)` 형태로 호출 시 내부적으로 `__call__`을 통해 실행

In [ ]:
# 3. Multi-Layer Perceptron 구성하기
import torch
import torch.nn as nn


class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        # 모델의 구성요소를 정의하는 부분입니다.

        # 🌟 PyTorch 모델을 구성할 때 항상 넣어줘야하는 코드입니다.
        super().__init__()

        # 여기에 Layer를 정의합니다.
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # 실제 데이터를 입력으로 받았을 때 __init__에서 정의한 레이어 순서대로 입력을 보내주는 코드입니다.
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


class SequentialMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()

        # 위에서 제작한 코드가 기본적인 형태입니다.
        # 하지만 매번 forward에서 정의한 layer들을 쓰면서 넘기기는 귀찮네요.
        # nn.Sequential을 이용하면 귀찮음을 해결할 수 있습니다.
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        # 귀찮음 해결 ☝️
        return self.layers(x)


# 실제로 작동하는 모델을 선언하려면 위에서 정의한 클래스를 객체로 만들어줘야합니다.
model = SequentialMLP(input_dim=20, hidden_dim=10, output_dim=1)

# 더미데이터를 만들어서 의도한대로 작동하는지 확인하는 작업입니다.
# 위에서 모델을 정의할 때 input_dim=20으로 만들었기 때문에 특성이 20개인 데이터를 만들어줍니다.
# 예시로 전체 데이터의 수는 40건이 있다고 가정합시다.
# 그렇다면 X는 다음과 같이 정의됩니다.
X = torch.ones(size=(40, 20))
y_pred = model(X)
y_pred.size()

지금 위에서 더미데이터를 생성할 때 텐서의 모양이 (40, 20)이었습니다. `nn.Linear`의 입력은 항상 (데이터의수, 입력차원수)로 받아줘야합니다. 왜냐하면 PyTorch는 데이터 학습을 배치처리하는데 효율적인 방법을 구현했기 때문입니다. 그래서 출력값 또한 (데이터수, 1)이 됩니다. `SequentialMLP`의 마지막 레이어가 `nn.Linear(hidden_dim, 1)`이기 때문에 출력 특성수가 하나이기 때문입니다.

### Concept Check: 객체에 `()`를 걸면 어떻게 될까?

위에서 조금 독특한 연산이 있습니다. 바로 `model(X)`와 `criterion(y_pred, y_true)`입니다. 단순 객체인데 왜 함수처럼 작동이 가능할까요? 정답은 예전에 설명한 magic method에 있습니다. 클래스에 `__call__`이 구현되어 있으면 생성한 객체를 함수처럼 사용할 수 있습니다.

그렇다면 PyTorch 모듈도 `__call__`를 커스텀하면 될까요? 위에 코드를 보셔서 아시겠지만 우리는 `forward`를 구현하였습니다. 실제로 `model(X)`를 수행하면 `forward`가 호출하는데, 이것은 PyTorch 구현에서 필요한 모든 동작이 `__call__`에 이미 정의되어 있기 때문입니다.

`nn.Module`의 `__call__` 내부는 대략 다음 순서로 동작합니다:
1. 입력 텐서에 대한 전처리(등록된 pre-forward hook 실행)
2. 실제 `forward` 메서드 호출
3. 출력 텐서에 대한 후처리(등록된 post-forward hook 실행)
4. 자동미분을 위한 그래프 연결 및 backward hook 등록

따라서 사용자는 오직 `forward`만 구현하면 되고, 모델을 호출할 때마다 PyTorch가 알아서 `__call__` ↔ `forward` 흐름을 관리해 줍니다. 결론적으로, 커스텀 레이어를 만들 때는 `__call__`이 아니라 `forward`만 정의하면 충분합니다.

## Step 3: Dataset과 DataLoader 활용

### Concept Check

이번에는 숫자 판독기를 만들어봅시다. `sklearn.datasets.load_digits`는 사람 손으로 쓰여진 0-9까지의 흑백필기 사진이 담겨져있습니다. 이미지이지만 (8, 8) 사이즈 밖에 되지 않아서 이를 일렬로 펴서 Multi-Layer Perceptron에 통과시켜 숫자를 예측해봅시다.

아래 코드는 데이터에 대한 샘플입니다.

In [ ]:
digits = load_digits()
X = digits.data.astype(np.float32)  # (n_samples, 64)
y = digits.target.astype(np.int64)

# 데이터를 한 번 살펴봅시다.
fig, ax = plt.subplots(figsize=(13, 5), nrows=2, ncols=5)
for i in range(10):
    ax[i // 5, i % 5].imshow(X[i].reshape(8, 8), cmap="gray")
    ax[i // 5, i % 5].set_title(f"Label: {y[i]}")
    ax[i // 5, i % 5].axis("off")
fig.tight_layout()

In [ ]:
# 아래는 난수를 고정하는 함수입니다.
# 재현성을 위해 구현되어 있습니다.
import random
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(seed=42)

### TODO 1: 데이터 분할 및 표준화

데이터를 분할하고 표준화해줍니다. 이번에는 검증데이터 (`X_valid`)과 테스트데이터 (`X_test`)까지 만들어줍시다.

**요구사항**:
1. `sklearn.model_selection.train_test_split`을 두 번 사용하여 학습데이터는 전체의 80%, 나머지 검증과 테스트데이터는 각각 10%씩 할당해주세요.
     - `X`, `y`를 우선 `X_train`, `X_temp`, `y_train`, `y_temp`로 나누고, `X_temp`, `y_temp`를 `X_valid`, `y_valid`, `X_test`, `y_test`로 분할해주세요.
     - 난수는 위에서 정의된 `SEED`를 활용해주세요.
     - stratify 하는것도 잊지 마시구요.
2. `sklearn.preprocessing.StandardScaler`를 통해 데이터를 표준화해주세요.


In [ ]:
# 1. `sklearn.model_selection.train_test_split`을 두 번 사용하여
# 학습데이터는 전체의 80%, 나머지 검증과 테스트데이터는 각각 10%씩 할당해주세요.
X_train, X_temp, y_train, y_temp = None
X_valid, X_test, y_valid, y_test = None

# 2. `sklearn.preprocessing.StandardScaler`를 통해 데이터를 표준화해주세요.
scaler = None
X_train = None
X_valid = None
X_test  = None

### TODO 2: PyTorch Dataset 만들기

batch 처리방식을 기억하시나요? 매번 X에서 인덱싱을 통해 데이터를 배치로 뽑아냈습니다. 여간 귀찮은게 아니었는데요, 이를 간편하게 만들어주는 방식이 PyTorch에 존재합니다.

- **요구사항**:
  1. TODO 1에서 만든 모든 데이터를 `torch.from_numpy`를 사용하여 `torch.Tensor`로 변환해주세요.
  2. 1에서 변환한 데이터를 PyTorch `TensorDataset`에 넣어주세요. (https://pytorch.org/docs/stable/data.html#torch.utils.data.TensorDataset)
  3. 위에서 만든 `TensorDataset`을 `DataLoader`로 변환시켜줍니다. 선언된 `BATCH_SIZE`를 활용해주세요.
     1. `DataLoader`에는 데이터를 꺼내는 순서인 `shuffle`이라는 인자가 있습니다. 훈련 데이터에는 이 값을 `True`로, 검증/테스트 데이터에서는 `False`로 넣어주세요.


**핵심 PyTorch 메서드 설명**:
- **TensorDataset(*tensors)**: 입력으로 동일한 첫 번째 차원 크기를 가진 텐서들을 받아 이를 하나의 데이터셋으로 묶어 인덱싱 시 튜플 형태로 샘플을 반환
- **DataLoader(dataset, batch_size, shuffle)**: 입력으로 Dataset 객체, 배치 크기, shuffle 여부를 받아 배치 단위로 데이터를 로드하는 이터레이터를 출력하며, 병렬 처리와 데이터 섞기를 지원

In [ ]:
# 1. 모든 데이터를 `torch.from_numpy`를 사용하여 `torch.Tensor`로 변환해주세요.
X_train_t = None
y_train_t = None
X_valid_t = None
y_valid_t = None
X_test_t  = None
y_test_t  = None

# 2. 변환된 데이터를 PyTorch `TensorDataset`에 넣어주세요.
from torch.utils.data import TensorDataset
train_ds = None
valid_ds = None
test_ds  = None

# 3. 위에서 만든 TensorDataset을 DataLoader로 변환시켜줍니다.
from torch.utils.data import DataLoader
BATCH_SIZE = 32

train_loader = None
valid_loader = None
test_loader  = None

### Concept Check: Dataset vs DataLoader

위에서 만든 Dataset과 DataLoader의 차이를 살펴봅시다.
- **Dataset**: 한 개를 정의한 것이기 때문에 단일 인덱싱이 가능합니다. Return 값은 `TensorDataset`에서 넣어준 데이터의 순서입니다. 즉 다음 코드는 `(X_train_t[0], y_train_t[0])`을 불러온 것이 됩니다.

In [ ]:
train_ds[0]

또한 `TensorDataset`에 몇 개의 데이터가 들어있는지도 알 수 있습니다.

In [ ]:
len(train_ds)

반면 **DataLoader**는 단일데이터 처리용이 아니기 때문에 인덱싱이 불가능합니다. 따라서 다음 코드는 에러가 나타납니다.

In [ ]:
# train_loader[0]  # 에러 발생!

그러면 단일 배치가 잘 불러졌는지 확인하는 방법에는 뭐가 있을까요? 힌트는 에러메시지에 있습니다. DataLoader를 Iterable 객체로 변환시켜주면 됩니다.

In [ ]:
next(iter(train_loader))

실제로 학습에서 활용할 때는 for loop에 넣어서 꺼내줍니다.

In [ ]:
for idx, batch in enumerate(train_loader):
    print(idx, batch)
    break

DataLoader도 `__len__` 함수가 있습니다. 전체 데이터수가 아닌, **batch의 개수**라는 점 기억하세요!

In [ ]:
len(train_loader)

## Step 4: MLP 모델 구현 및 학습

### Concept Check

데이터를 불러오는 것까지 잘 성공했네요. 이제는 학습 부분으로 넘어갑시다. 모델을 정의해주고 파라미터를 업데이트 하는 부분까지 정리해주면 마무리됩니다. 다음 세 가지를 하나씩 달성해주세요.

1. **모델 정의**: 기본적인 MLP 모델을 지시사항에 따라 구현해주세요.
2. **배치처리 정의**: 하나의 DataLoader와 모델이 주어졌을 때 처리하는 함수를 정의합니다.
3. **학습 파이프라인 구축**: 1, 2를 결합하여 학습파이프라인을 구축합니다.

**핵심 PyTorch 메서드 설명**:
- **nn.Dropout(p)**: 입력으로 드롭아웃 확률 p를 받아 학습 시 각 뉴런을 확률 p로 0으로 만들어 과적합을 방지하는 정규화 레이어를 출력하며, 평가 시에는 자동으로 비활성화
- **Module.train()**: 입력 없이 호출되며 모델을 학습 모드로 전환하여 Dropout과 BatchNorm 같은 레이어가 학습 동작을 수행하도록 설정
- **Module.eval()**: 입력 없이 호출되며 모델을 평가 모드로 전환하여 Dropout을 비활성화하고 BatchNorm이 학습된 통계를 사용하도록 설정

### TODO 3: Multi-Layer Perceptron 구현하기

가장 기본적인 MultiLayer Perceptron을 구현해주세요.

- **요구사항**:
  - 계층은 총 3개입니다: 입력표현 → 은닉표현1 → 은닉표현2 → 출력표현
  - 각 은닉표현은 `nn.Linear`를 통과한 뒤 `nn.ReLU`를 거치고 `nn.Dropout`을 거치도록 설계해주세요.
  - 제가 정의해둔 `__init__` 안의 인자에 맞춰서 작업해주세요

In [ ]:
class MLP(nn.Module):
    def __init__(self,
                 input_dim: int,
                 num_classes: int,
                 hidden_dims: Tuple[int] = (128, 64),
                 dropout: float = 0.2):
        super().__init__()
        h1, h2 = hidden_dims
        # forward가 동작되도록 여기에 구현해주세요.
        self.net = None

    def forward(self, x):
        return self.net(x)

### TODO 4: 학습 함수 작성

DataLoader, Model, Optimizer가 주어졌을 때 학습데이터에 대해 model을 업데이트하는 코드를 작성해주세요.

- **요구사항**:
  1. 모델이 학습모드로 들어가게 설정해야합니다. (`model.train()`)
  2. 위에서 배운 파라미터 업데이트 방식을 구현해주세요:
     1. 데이터를 가속화기기로 보내주세요.
     2. 모델의 출력값인 `logits`를 계산합니다.
     3. 미분값 지워주기 (`zero_grad`)
     4. `torch.nn.functional.cross_entropy`를 통해 `loss` 계산해주기
     5. `loss`를 기준으로 역전파를 수행합니다.
     6. 파라미터 업데이트를 진행해주세요

**핵심 PyTorch 메서드 설명**:
- **F.cross_entropy(input, target)**: 입력으로 로짓(logits)과 정수 레이블을 받아 소프트맥스를 적용한 후 음의 로그 우도를 계산하여 다중 클래스 분류의 손실값을 스칼라 텐서로 출력
- **Tensor.argmax(dim)**: 입력으로 차원 인덱스를 받아 해당 차원에서 최댓값의 인덱스를 반환하며, 분류 모델의 예측 클래스를 얻을 때 사용

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    running_loss = 0.0
    correct = 0
    total = 0

    # 1. 모델이 학습모드로 들어가게 설정해야합니다.
    model.train()

    # 2. 위에서 배운 파라미터 업데이트 방식을 구현해주세요:
    for xb, yb in loader:
        # 2.1 데이터를 가속화기기로 보내주세요.
        xb, yb = None

        # 2.2 모델의 출력 만들어주기
        logits = None

        # 2.3 미분값 지워주기 (`zero_grad`)


        # 2.4 `torch.nn.functional.cross_entropy`를 통해 `loss` 계산해주기
        loss = None

        # 2.5 `loss`를 기준으로 역전파를 수행합니다.


        # 2.6 파라미터 업데이트를 진행해주세요


        # 파라미터 업데이트가 끝났으면 몇 가지를 기록해줍니다.
        running_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    # 위에서 잘 계산된 값들을 정리해서 반환합니다.
    avg_loss = running_loss / total
    acc = correct / total
    return avg_loss, acc

### TODO 5: 평가 함수 작성

이번에는 모델과 검증/테스트데이터가 입력으로 들어왔을 때 정확도를 계산해주는 코드를 작성해주세요.

- **요구사항**:
  1. 모델이 추론할 때는 추론모드로 설정해주어야합니다. (`model.eval()`)
  2. 추론 시에는 미분그래프를 생성하지 않도록 컨텍스트를 생성해주셔야합니다. (`torch.no_grad()`)
  3. 추론 과정 루프를 다음 순서에 맞춰 작성해주세요.
     1. 데이터를 가속화기기로 보내주세요.
     2. 모델의 출력값인 `logits`를 계산합니다.
     3. `torch.nn.functional.cross_entropy`를 통해 `loss` 계산해주기

In [ ]:
def evaluate(model, loader, device):
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    # 1. 모델이 추론할 때는 추론모드로 설정해주어야합니다.
    model.eval()
    # 2. 추론 시에는 미분그래프를 생성하지 않도록 컨텍스트를 생성해주셔야합니다.
    with torch.no_grad():
        # 3. 추론 과정 루프를 다음 순서에 맞춰 작성해주세요.
        for xb, yb in loader:
            # 3.1 데이터를 가속화기기로 보내주세요.
            xb, yb = None

            # 3.2 모델의 출력값인 `logits`를 계산합니다.
            logits = None

            # 3.3 `torch.nn.functional.cross_entropy`를 통해 `loss` 계산해주기
            loss = None

            # 기록
            running_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

            all_preds.append(preds.cpu())
            all_targets.append(yb.cpu())

    # 위에서 잘 계산된 값들을 정리해서 반환합니다.
    avg_loss = running_loss / total
    acc = correct / total
    preds_cat = torch.cat(all_preds).numpy()
    targets_cat = torch.cat(all_targets).numpy()
    return avg_loss, acc, preds_cat, targets_cat

### Concept Check: `model.train()` vs `model.eval()`

PyTorch에서 `model.train()`과 `model.eval()`은 모델의 "학습/평가 모드"를 전환하는 메서드입니다. 이 메서드는 파라미터 값을 바꾸는 것이 아니라, 레이어 동작 방식을 바꾸는 역할을 합니다.

**주로 영향을 받는 레이어**:
- **Dropout**: 학습 중에만 일부 뉴런을 확률적으로 0으로 만듭니다. 추론모드에서는 모든 뉴런을 사용합니다.
- **BatchNorm**: 학습 중에는 현재 배치의 평균과 분산으로 정규화합니다. 추론모드에서는 학습된 running mean/variance를 사용합니다.

정확한 모델의 추론결과를 보려면 추론 시작 전에 `model.eval()`을 호출해야합니다.

**핵심 PyTorch 메서드 설명**:
- **torch.save(obj, path)**: 입력으로 Python 객체(모델 state_dict 등)와 파일 경로를 받아 객체를 pickle 형식으로 직렬화하여 저장
- **torch.load(path, map_location)**: 입력으로 저장된 파일 경로와 로드할 장치를 받아 역직렬화된 Python 객체를 반환
- **Module.state_dict()**: 입력 없이 호출되며 모델의 모든 학습 가능한 파라미터를 OrderedDict 형태로 반환
- **Module.load_state_dict(state_dict)**: 입력으로 state_dict를 받아 모델의 파라미터를 해당 값으로 로드

### 학습 실행

모든 것을 하나로 합쳐봅시다! 여러분이 코드를 잘 짰다면 아래 코드가 잘 작동할겁니다. 에러가 뜬다면 에러메시지를 확인해서 잘 구현되도록 맞춰주세요.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model = MLP(
    input_dim=64,
    num_classes=10,
    hidden_dims=(128, 64),
    dropout=0.2
).to(DEVICE)

# 이번에는 지난 과제2에서 배웠던 Adam을 사용해봅시다.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

best_val_loss = math.inf
best_val_acc = -1.0
epochs_no_improve = 0

checkpoint_path = "model.ckpt"
earlystop_patience = 5

train_losses, train_accs = [], []
valid_losses, valid_accs = [], []
for epoch in range(1, 201):
    t0 = time.time()
    # 여러분이 정의한 `train_one_epoch`와 `evaluate`입니다.
    train_loss, train_acc = train_one_epoch(model=model,
                                            loader=train_loader,
                                            optimizer=optimizer,
                                            device=DEVICE)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    val_loss, val_acc, _, _ = evaluate(model=model,
                                       loader=valid_loader,
                                       device=DEVICE)
    valid_losses.append(val_loss)
    valid_accs.append(val_acc)

    dt = time.time() - t0
    print(
        f"Epoch {epoch:03d} | "
        f"train loss {train_loss:.4f} acc {train_acc*100:5.2f}% | "
        f"val loss {val_loss:.4f} acc {val_acc*100:5.2f}% | "
        f"{dt:.1f}s"
    )

    # 개선 시 체크포인트 저장
    improved = (val_loss < best_val_loss) or (val_acc > best_val_acc)
    if improved:
        best_val_loss = min(best_val_loss, val_loss)
        best_val_acc = max(best_val_acc, val_acc)
        torch.save({
            'model_state_dict': model.state_dict(),
            'input_dim': 64,
            'num_classes': 10,
        }, checkpoint_path)
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    # 조기 종료
    if epochs_no_improve >= earlystop_patience:
        print(f"Early stopping at epoch {epoch} (no improve {earlystop_patience})")
        break

# 베스트 모델 로드 후 테스트 평가
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded best checkpoint: val_best_acc={best_val_acc*100:.2f}% val_best_loss={best_val_loss:.4f}")

test_loss, test_acc, y_pred, y_true = evaluate(model=model,
                                               loader=test_loader,
                                               device=DEVICE)
print("\n==== Test Result ====")
print(f"Test loss: {test_loss:.4f}")
print(f"Test acc : {test_acc*100:.2f}%")

# 분류 리포트 / 혼동행렬
print("\nClassification report:")
print(classification_report(y_true, y_pred, digits=4))

print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

### 학습 과정 시각화

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

fig, ax1 = plt.subplots(figsize=(8, 5))

# 첫 번째 y축: Loss
color1 = "tab:red"
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
sns.lineplot(data=train_losses, label="Train Loss", ax=ax1, color="red")
sns.lineplot(data=valid_losses, label="Valid Loss", ax=ax1, color="orange")
ax1.tick_params(axis='y')

# 두 번째 y축: Accuracy
ax2 = ax1.twinx()
color2 = "tab:blue"
ax2.set_ylabel("Accuracy")
sns.lineplot(data=train_accs, label="Train Accuracy", ax=ax2, color="blue")
sns.lineplot(data=valid_accs, label="Validation Accuracy", ax=ax2, color="skyblue")
ax2.tick_params(axis='y')

# 범례 통합
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="center right")
if ax2.legend_:
    ax2.legend_.remove()

plt.title("Training Progress", size="large")
plt.tight_layout()
plt.show()

### 참고: 학습이 길어지면?

현재 제작한 코드는 loss를 중간중간 계산하긴 하지만, 전체 학습과정에서 산출된 loss를 보관하고 있는 `train_losses` 와 같은 변수들은 오류가 발생하면 저장되지 않습니다. 그리고 시각화를 중간중간 가능하게 하는 코드가 있기는 하지만 많이 번거롭습니다. 이런 경우를 방지하기 위해 `wandb`라는 라이브러리들이 존재합니다. 이는 서버에 중간중간 기록을 보내 플롯을 그려주는 라이브러리입니다. 한 번 살펴보시고 도입해보세요.
- [Weight and Biases Colaboratory Tutorial](https://colab.research.google.com/github/wandb/examples/blob/master/colabs/intro/Intro_to_Weights_%26_Biases.ipynb)

### 참고: `sklearn.classification`에 있는 모듈로도 계산이 될까요?

우리가 지금 파리 잡는데 도끼를 든 격이 아닌가 생각해볼 필요가 있습니다. 8×8 사이즈의 이미지 정도면 사실 머신러닝 모델로도 어느 정도 성능을 달성할 수 있습니다. 한 번 시도해보세요.

## **학생용 자가 체크리스트**

- [ ] PyTorch 텐서의 기본 연산을 수행할 수 있다.
- [ ] Autograd를 활용해 자동 미분을 수행하고 기울기를 확인할 수 있다.
- [ ] `nn.Linear`, 손실 함수, 옵티마이저를 활용해 간단한 선형회귀를 구현할 수 있다.
- [ ] `nn.Module`을 상속받아 커스텀 신경망 클래스를 정의할 수 있다.
- [ ] `nn.Sequential`을 활용해 MLP 모델을 구성할 수 있다.
- [ ] `forward` 메서드와 `__call__`의 관계를 설명할 수 있다.
- [ ] NumPy 배열과 PyTorch 텐서 간 변환을 수행할 수 있다.
- [ ] GPU로 텐서를 이동시켜 가속화된 연산을 수행할 수 있다.
- [ ] `TensorDataset`과 `DataLoader`를 활용해 배치 처리 파이프라인을 구성할 수 있다.
- [ ] `train()`/`eval()` 모드 전환의 필요성을 설명할 수 있다.
- [ ] 학습 루프와 평가 루프를 작성할 수 있다.
- [ ] Early Stopping을 적용하여 최적 모델을 저장하고 불러올 수 있다.
- [ ] CrossEntropyLoss와 옵티마이저를 활용해 분류 모델을 학습시킬 수 있다.
- [ ] 학습 과정을 시각화하고 성능을 평가할 수 있다.

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성청년SW·AI아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.